In [12]:
import os
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import src.config as cfg
import src.data_prep as dp
import src.power_flow as pfo
 


In [13]:
df_raw = dp.load_and_prepare_data()
df_sort=dp.sort_ship_columns(df_raw, start_idx=6, First_ship=0, Last_ship=70)
df_sort.head()

,private_MWh,price_EUR_MWh,business_MWh,CO2_g_MWh,t2m_C,radiation_solaire_factor,AIDANOVA,MSC EURIBIA,IONA,NORWEGIAN PRIMA,...,EUROPA,SEABOURN VENTURE,STAR PRIDE,DEUTSCHLAND,HAMBURG,CORINTHIAN,WORLD VOYAGER,HANSEATIC SPIRIT,HEBRIDEAN SKY,NOORDERLICHT
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.202830,26.80,0.945704,4730.0,0.75967,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 01:00:00,0.197758,32.80,0.924756,4600.0,0.66458,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 02:00:00,0.192327,31.14,0.927263,3920.0,0.58303,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 03:00:00,0.187711,32.80,0.928286,3790.0,0.46475,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 04:00:00,0.186129,33.10,0.963000,4050.0,0.43105,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
network=pfo.create_pypsa_network(df_sort,50,True)
pv_size=50
network.generators.loc["solar", "p_nom"] = 10
network.optimize()

/var/folders/_p/lntvskx96x323cvnklm36kvh0000gn/T/ipykernel_49624/2047902086.py:4: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  network.optimize()
Index(['bus_main', 'bus_ship', 'bus_solar', 'bus_battery'], dtype='str', name='name')
Index(['link_shore_power', 'link_solar_main', 'link_battery_to_main',
       'link_main_to_battery', 'link_solar_to_battery'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 5/5 [00:00<00:00, 713.49it/s]
INFO:linopy.io: Writing time: 0.05s


Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-bi7nhra3 has 219600 rows; 87840 cols; 333791 nonzeros
Coefficient ranges:
  Matrix  [9e-01, 1e+00]
  Cost    [1e-02, 4e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e-04, 1e+06]
Presolving model
31123 rows, 67017 cols, 120478 nonzeros  0s
27110 rows, 54220 cols, 94884 nonzeros  0s
Dependent equations search running on 22287 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
22287 rows, 49397 cols, 85186 nonzeros  0s
Presolve reductions: rows 22287(-197313); columns 49397(-38443); nonzeros 85186(-248605) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.1s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 87840 primals, 219600 duals
Objective: 3.06e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


      31489     3.0565005516e+06 Pr: 0(0); Du: 0(2.26804e-13) 0.4s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-bi7nhra3
Model status        : Optimal
Simplex   iterations: 31489
Objective value     :  3.0565005516e+06
P-D objective error :  2.2090909023e-15
HiGHS run time      :          0.44


('ok', 'optimal')

In [18]:
network.generators

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
grid_import,bus_main,Slack,,1000000.0,0.0,False,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,1000000.0
solar,bus_solar,Slack,,10.0,0.0,False,0.0,inf,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,10.0


In [20]:
a=network.statistics()
autoconso=a['Energy Balance']['Generator']['Solar']/(a['Energy Balance']['Generator']['Grid']+a['Energy Balance']['Generator']['Solar'])
autoconso

np.float64(0.24308960986908384)

### Pareto

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Configuration du style Matplotlib ---
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
# 1. Chargement du fichier CSV
# Remplacez 'resultats_scenarios.csv' par le chemin exact de votre fichier
df_all = pd.read_csv('/Users/eliotsouthon/Desktop/Alesund_Simulation_cost/independence_pareto_all_scenarios.csv')

# 2. Calcul du Taux de pénétration renouvelable (%)
df_all["Taux_Renouvelable"] = np.where(
    df_all["E_load"] > 0, 
    (df_all["E_pv"] / df_all["E_load"]) * 100.0, 
    0
)

# 3. Séparation des scénarios pour faciliter les tracés
df_s0 = df_all[df_all["Scenario"].str.contains("S0")].copy()
df_s1 = df_all[df_all["Scenario"].str.contains("S1")].copy()
df_s2 = df_all[df_all["Scenario"].str.contains("S2")].copy()

# Si le DataFrame S2 est vide (le bout de CSV fourni ne contient que S0 et S1),
# le code ne plantera pas mais les graphiques S2 seront vides.
print(f"Lignes S0: {len(df_s0)} | Lignes S1: {len(df_s1)} | Lignes S2: {len(df_s2)}")
df_all[["Scenario", "PV_MW", "Bat_MWh", "Incremental_Cost", "Taux_Renouvelable"]].head()

In [ ]:
def extract_pareto_max_min(df_res: pd.DataFrame,
                           x_col: str = "Taux_Renouvelable",
                           y_col: str = "Incremental_Cost") -> pd.DataFrame:
    """Extrait le front de Pareto pour un problème Maximiser(X) - Minimiser(Y)."""
    if df_res.empty:
        return pd.DataFrame(columns=df_res.columns)
        
    # Trier par coût croissant (le moins cher en premier)
    s = df_res.sort_values(by=y_col, ascending=True).reset_index(drop=True)
    keep, best_x = [], -np.inf
    
    # Garder le point uniquement s'il améliore strictement le taux renouvelable
    for i, row in s.iterrows():
        if row[x_col] > best_x:
            keep.append(i)
            best_x = row[x_col]
            
    # Retourner trié par l'axe X
    return s.loc[keep].sort_values(by=x_col).reset_index(drop=True)

pareto_s1 = extract_pareto_max_min(df_s1)
pareto_s2 = extract_pareto_max_min(df_s2)

if not df_s2.empty:
    opt_s2 = df_s2.loc[df_s2["Incremental_Cost"].idxmin()]
    print("Configuration optimale S2 (Coût minimum) :")
    print(f"  • PV                 = {opt_s2['PV_MW']:.0f} MW")
    print(f"  • Battery            = {opt_s2['Bat_MWh']:.0f} MWh")
    print(f"  • Incremental cost   = {opt_s2['Incremental_Cost']/1e6:.3f} M€/an vs S0")
    print(f"  • Taux Renouvelable  = {opt_s2['Taux_Renouvelable']:.2f} %")

In [ ]:
def extract_pareto_max_min(df_res: pd.DataFrame,
                           x_col: str = "Taux_Renouvelable",
                           y_col: str = "Incremental_Cost") -> pd.DataFrame:
    """Extrait le front de Pareto pour un problème Maximiser(X) - Minimiser(Y)."""
    if df_res.empty:
        return pd.DataFrame(columns=df_res.columns)
        
    # Trier par coût croissant (le moins cher en premier)
    s = df_res.sort_values(by=y_col, ascending=True).reset_index(drop=True)
    keep, best_x = [], -np.inf
    
    # Garder le point uniquement s'il améliore strictement le taux renouvelable
    for i, row in s.iterrows():
        if row[x_col] > best_x:
            keep.append(i)
            best_x = row[x_col]
            
    # Retourner trié par l'axe X
    return s.loc[keep].sort_values(by=x_col).reset_index(drop=True)

pareto_s1 = extract_pareto_max_min(df_s1)
pareto_s2 = extract_pareto_max_min(df_s2)

if not df_s2.empty:
    opt_s2 = df_s2.loc[df_s2["Incremental_Cost"].idxmin()]
    print("Configuration optimale S2 (Coût minimum) :")
    print(f"  • PV                 = {opt_s2['PV_MW']:.0f} MW")
    print(f"  • Battery            = {opt_s2['Bat_MWh']:.0f} MWh")
    print(f"  • Incremental cost   = {opt_s2['Incremental_Cost']/1e6:.3f} M€/an vs S0")
    print(f"  • Taux Renouvelable  = {opt_s2['Taux_Renouvelable']:.2f} %")

In [ ]:
def plot_comparative_pareto(df_s0, df_s1, df_s2, pareto_s2):
    fig, ax = plt.subplots(figsize=(12, 7))

    # --- S2: Nuage de points (couleur = PV, taille = Batterie) ---
    size_mult, base = 18, 25
    if not df_s2.empty:
        sc = ax.scatter(df_s2["Taux_Renouvelable"], df_s2["Incremental_Cost"] / 1e6,
                        c=df_s2["PV_MW"], cmap="viridis",
                        s=df_s2["Bat_MWh"] * size_mult + base,
                        alpha=0.45, edgecolor="white", linewidth=0.3)
        
        cbar = plt.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label("Capacité PV installée (MW)", rotation=270, labelpad=18)

        # Front de Pareto S2
        h_s2, = ax.plot(pareto_s2["Taux_Renouvelable"], pareto_s2["Incremental_Cost"] / 1e6,
                        color="red", linewidth=2.5, linestyle="--", label="S2 – Pareto Front")

        # Optimum S2
        opt = df_s2.loc[df_s2["Incremental_Cost"].idxmin()]
        h_opt = ax.scatter(opt["Taux_Renouvelable"], opt["Incremental_Cost"] / 1e6,
                           color="red", marker="*", s=420, edgecolor="black", zorder=7)

    # --- S1: Ligne (PV uniquement) ---
    if not df_s1.empty:
        s1_sorted = df_s1.sort_values("Taux_Renouvelable")
        h_s1, = ax.plot(s1_sorted["Taux_Renouvelable"], s1_sorted["Incremental_Cost"] / 1e6,
                        color="tab:blue", linewidth=2.2, marker="o", markersize=5, label="S1 – PV only")

    # --- S0: Référence ---
    if not df_s0.empty:
        s0 = df_s0.iloc[0]
        h_s0 = ax.scatter(s0["Taux_Renouvelable"], 0,
                          color="black", marker="X", s=220, zorder=6, label="S0 – Référence")
        ax.axhline(0, color="gray", linewidth=0.8, linestyle=":", alpha=0.7)

    # --- Configuration des axes ---
    ax.set_xlabel(r"Taux de pénétration renouvelable  $(E_{pv} / E_{load}) \times 100$  (%)  $\rightarrow$ plus d'autonomie", fontsize=12)
    ax.set_ylabel(r"Coût Incrémental Annualisé vs $\mathcal{S}_0$ (M€/an)", fontsize=12)
    ax.set_title("Pareto Bi-objectif : Taux Renouvelable vs Coût Incrémental", fontsize=13)
    
    # Légendes (ajustées selon ce qui existe dans les données)
    handles, labels = ax.get_legend_handles_labels()
    main_leg = ax.legend(handles, labels, loc="upper left", framealpha=0.92, title="Scénarios")
    ax.add_artist(main_leg)

    plt.tight_layout()
    plt.show()

plot_comparative_pareto(df_s0, df_s1, df_s2, pareto_s2)

In [ ]:
def plot_penetration_heatmap(df_s2):
    if df_s2.empty:
        print("Aucune donnée S2 disponible pour générer la Heatmap.")
        return

    pivot = df_s2.pivot(index="Bat_MWh", columns="PV_MW", values="Taux_Renouvelable")

    fig, ax = plt.subplots(figsize=(14, 5))
    
    # Colormap: Vert = Bon (Taux élevé), Rouge = Moins bon
    im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values,
                       cmap="RdYlGn", shading="auto")
    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label("Taux de pénétration renouvelable (%)")

    # Iso-contours
    lo = int(np.floor(pivot.values.min() / 10.0)) * 10
    hi = int(np.ceil (pivot.values.max() / 10.0)) * 10
    cs = ax.contour(pivot.columns, pivot.index, pivot.values,
                    levels=np.arange(lo, hi + 1, 10),
                    colors="black", linewidths=0.7, alpha=0.6)
    ax.clabel(cs, inline=True, fmt="%d%%", fontsize=8)

    # Point optimal
    opt = df_s2.loc[df_s2["Incremental_Cost"].idxmin()]
    ax.scatter(opt["PV_MW"], opt["Bat_MWh"],
               marker="*", s=420, color="red", edgecolor="black", zorder=5,
               label=(f"Optimum économique\nPV={opt['PV_MW']:.0f} MW, "
                      f"Bat={opt['Bat_MWh']:.0f} MWh"))

    ax.set_xlabel(r"Capacité PV installée (MW)")
    ax.set_ylabel(r"Capacité Batterie installée (MWh)")
    ax.set_title("S2 – Taux de pénétration renouvelable selon le dimensionnement")
    ax.legend(loc="upper left", framealpha=0.95)
    
    plt.tight_layout()
    plt.show()

plot_penetration_heatmap(df_s2)